**24 | Multi-Seed Stability and Class-Imbalance Loss Ablation**

**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**Research objective:** Verify whether Stage 23's leading temporal architectures remain competitive under three initialisation seeds, and whether Focal Loss or Class-Balanced Cross-Entropy offers a controlled improvement over the existing inverse-square-root weighted Cross-Entropy for 1D-ResNet. The 29-person HARTH development cohort, original 500-cap source rows and three GroupKFold partitions are **fixed and shared** between conditions. No original protected Test subject is loaded or scored.

This is an exploratory robustness study. A change of loss may improve one class and impair another. Do not select a favourable seed, infer clinical activity accuracy, or declare architecture superiority from these dependent development folds.

**01. Research Questions, Experimental Controls and Colab GPU Setup**

Stage 23 provided observed mean held-out-fold Macro F1 values 0.8328 (1D-ResNet), 0.8138 (TCN) and 0.8094 (Random Forest). Yet the Random Forest achieved Shuffling F1 of 0.5032, compared with 0.3962 for 1D-ResNet and 0.3392 for TCN. The first experiment repeats 1D-ResNet and TCN using three predeclared seeds (`42`, `123`, `2026`) under identical inverse-square-root weighted Cross-Entropy; the second fixes 1D-ResNet and compares Weighted CE, Focal Loss (gamma 2) and Class-Balanced CE (beta 0.9999) across the same seeds and folds. Shared Weighted CE runs are performed **only once**. GPU compute is required for the substantial training workload.

In [ ]:
# ==================== 01. Scientific Protocol and GPU Verification ====================

# 1. Resolve repository and import the existing reproducible research tools.
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
root=Path.cwd().resolve()
while not (root/"code/python/src/core/data_loading.py").is_file():
    if root==root.parent: raise RuntimeError("Run inside the HMI repository.")
    root=root.parent
sys.path.insert(0,str(root/"code/python/src"))
sys.path.insert(0,str(root/"studies/wearable_activity_recognition/src"))
from harth_data import HARTH_SUBJECTS, ACTIVITY_NAMES, SENSOR_COLUMNS, prepare_harth_window_cohort
from core.temporal_benchmark import make_benchmark_plan
from core.robustness_experiments import (
    ROBUST_MODELS, LOSS_METHODS, run_robustness_job,
    summarise_robustness_results, calculate_loss_weights
)

# 2. Require CUDA and use full float32 calculations as in Stage 23.
if not torch.cuda.is_available():
    raise RuntimeError("Enable Colab GPU in Runtime > Change runtime type.")
torch.set_float32_matmul_precision("highest")
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False
DEVICE="cuda"
SEEDS=(42,123,2026)
CLASSES=tuple(range(1,9))
RESERVED_TEST={"S023","S032"}
DEVELOPMENT=sorted(set(HARTH_SUBJECTS)-RESERVED_TEST)
CACHE_DIR=root/"data/processed/harth_stage24_robustness"
MAX_STEPS=400
assert len(DEVELOPMENT)==29 and len(SEEDS)==3

# 3. Print the exact research environment and fixed scientific controls.
print("\n"+"="*70)
print("STAGE 24 | MULTI-SEED AND LOSS ROBUSTNESS")
print("="*70)
print("GPU:",torch.cuda.get_device_name(0))
print("PyTorch:",torch.__version__,"| CUDA:",torch.version.cuda)
print("Seeds:",SEEDS,"| Group folds: 3 | Updates per job:",MAX_STEPS)
print("Protected source subjects:",sorted(RESERVED_TEST))
print("="*70)

**02. Reconstruct and Fingerprint the Original Matched Source Cohort**

Use the official pinned HARTH v2.0 data and exactly the established 500-cap chronological source-window protocol. Each source recording is windowed independently with 100 samples, 50 stride, 50 Hz, strict annotation purity and timestamp continuity. Label codes outside the eight-class task are excluded only after segmentation. The resulting 14,201 windows are split into three participant-independent folds. All conditions use the same source SHA256; this prevents treating a new sample as a paired rerun.

In [ ]:
# ==================== 02. Reconstruct Stage 23 Source-Windows and Folds ====================

# 1. Prepare source-aligned, documented activity windows without original Test.
X,y,metadata,overview=prepare_harth_window_cohort(
    subjects=DEVELOPMENT,data_dir=root/"data/raw/harth_v2",
    task_classes=CLASSES,max_windows_per_subject=500,
    window_size=100,step_size=50,sampling_rate=50,
)
assert X.shape==(14201,6,100)
assert not(set(metadata["subject_id"])&RESERVED_TEST)

# 2. Freeze original development groups and the three paired held-out folds.
plan=make_benchmark_plan(X,y,metadata["subject_id"].to_numpy(),
                         CLASSES,n_splits=3,forbidden_subjects=RESERVED_TEST)
print("Development source SHA256:",plan["fingerprint"])
display(overview[["subject_id","eligible_windows","sampled_windows"]])
assert len(plan["folds"])==3

**03. Predeclare Unique Model–Loss–Seed Experiments**

The complete unique experiment grid contains 12 distinct model/loss/seed conditions and 36 Fold-specific training jobs: six conditions for Multi-Seed (ResNet and TCN × three seeds) plus six additional ResNet Focal/Class-Balanced conditions (two losses × three seeds), times three folds. No duplicate Weighted CE training occurs. We retain Random Forest's Stage 23 results as an informative *historical reference*, not a new multi-seed RF experiment. Each new job uses a fresh model, Fold Train-only normalisation and loss weights, 400 steps and a fixed last-step prediction. Results are cached only after successful completion.

In [ ]:
# ==================== 03. Unique Experimental Matrix and Control Plan ====================

# 1. Reuse the shared weighted-loss baseline rather than duplicate its jobs.
EXPERIMENTS=[
    (model,"weighted_ce",seed)
    for model in ("1D-ResNet","TCN") for seed in SEEDS
]+[
    ("1D-ResNet",loss,seed)
    for loss in ("focal","class_balanced") for seed in SEEDS
]
assert len(EXPERIMENTS)==12 and len(set(EXPERIMENTS))==12
job_grid=pd.DataFrame([
    {"architecture":arch,"loss":loss,"seed":seed,"fold":fold}
    for arch,loss,seed in EXPERIMENTS for fold,_,_ in plan["folds"]
])
display(job_grid)
print("Distinct conditions:",len(EXPERIMENTS))
print("Unique model/fold training jobs:",len(job_grid))

# 2. Record all preselected fixed-batch training hyperparameters.
conditions=pd.DataFrame([
    {"setting":"Train source-window cap","value":500},
    {"setting":"Optimizer updates","value":MAX_STEPS},
    {"setting":"Batch size","value":128},
    {"setting":"Learning rate","value":0.001},
    {"setting":"Focal gamma","value":2.0},
    {"setting":"Class-balanced beta","value":0.9999},
    {"setting":"Primary metric","value":"Fixed 8-class Macro F1"},
])
display(conditions)

**04. Training-Only Class Weights and Focal-Loss Definition**

Focal Loss multiplies ordinary per-example cross-entropy by `(1-p_true)^gamma` and the original Train-only inverse-square-root class alpha; its reduction normalises by summed alpha per mini-batch. Weighted CE uses the unchanged Stage 23 inverse-square-root weighting; Class-Balanced CE instead uses effective-number class weighting with fixed beta. No Validation class prevalence is used in optimisation. Class-Balanced Loss and Focal Loss are **alternatives**, not proven remedies for rare activity generalisation. Their scales and gradient dynamics can differ.

In [ ]:
# ==================== 04. Inspect Loss-Weight Differences on Fold Train ====================

# 1. Use only the first Fold's Train labels for illustrative loss-weight tables.
_,first_train,first_validation=plan["folds"][0]
count_train=pd.Series(y[first_train]).value_counts().reindex(CLASSES,fill_value=0)
loss_weights=pd.DataFrame({
    "class_code":CLASSES,
    "activity":[ACTIVITY_NAMES[c] for c in CLASSES],
    "train_windows":count_train.to_numpy(),
})
for mode in LOSS_METHODS:
    loss_weights[mode]=calculate_loss_weights(y[first_train],CLASSES,method=mode)
display(loss_weights.round(4))

# 2. Verify no held-out original participant contributes to these weights.
assert not(set(plan["groups"][first_train])&set(plan["groups"][first_validation]))
print("Loss-weight calibration source: FOLD TRAIN ONLY")

**05. Run 36 Source-Fingerprinted GPU Jobs with Safe Resumption**

Each unique `(architecture, loss, seed, Fold)` is trained once, then its held-out source-row IDs, true/predicted codes, observed metrics, runtime, and complete configuration fingerprint are written atomically to the ignored local project cache. This allows a stopped Colab run to restart when files remain available; a full Colab Runtime reset can erase them. The experiment code rejects incomplete or mismatched cached predictions. Large training runs are not represented as complete until all 36 required jobs finish.

In [ ]:
# ==================== 05. Real GPU Multi-Seed and Loss Experiments ====================

# 1. Run every predeclared condition on the same held-out group partitions.
jobs=[]
for architecture,loss,seed in EXPERIMENTS:
    print("\n"+"="*70)
    print(f"MODEL: {architecture} | LOSS: {loss} | SEED: {seed}")
    print("="*70)
    for fold,train_rows,validation_rows in plan["folds"]:
        result=run_robustness_job(
            plan,architecture,fold,seed,loss,CACHE_DIR,
            device=DEVICE,max_steps=MAX_STEPS,
            batch_size=128,learning_rate=1e-3,
            focal_gamma=2.0,beta=0.9999,
        )
        jobs.append(result)

# 2. Refuse to report partial trials as a complete experiment.
assert len(jobs)==len(job_grid)==36
print("\n"+"="*70)
print("ALL 36 MATCHED DEVELOPMENT ROBUSTNESS JOBS FINISHED")
print("Cache location:",CACHE_DIR)
print("="*70)

**06. Aggregate Exact Out-of-Fold Evidence for Every Condition**

Each model/loss/seed condition must predict all 14,201 bounded development source windows exactly once across three folds. Class-support tables are derived from real prediction rows and keep the fixed eight-class denominator. Separate fold metrics, participant-level Macro F1 and pooled class-level F1 are calculated. Overlapping windows are correlated; the independent observational grouping in these analyses is the participant, not each window.

In [ ]:
# ==================== 06. Complete Held-Out Evidence and Metric Reconciliation ====================

# 1. Rebuild all metrics from source-verified prediction/label pairs.
fold_results,subject_results,class_results,predictions=(
    summarise_robustness_results(plan,jobs,EXPERIMENTS)
)
assert len(fold_results)==36
display(fold_results[[
    "architecture","loss_method","seed","fold","macro_f1",
    "balanced_accuracy","train_seconds","validation_windows"
]].round(4))

# 2. Show independent participant and fixed-task class coverage.
assert len(subject_results)==29*len(EXPERIMENTS)
assert len(class_results)==len(CLASSES)*len(EXPERIMENTS)
print("Unique development participants:",subject_results["subject_id"].nunique())
print("True held-out source rows per condition:",len(y))

**07. Architecture Multi-Seed Robustness: 1D-ResNet versus TCN**

Hold the loss constant at inverse-square-root weighted Cross-Entropy to examine training-randomness sensitivity. Report each of the three seeds and each of the three held-out folds, plus equal-seed means and the across-seed spread. Three seeds provide a preliminary stability picture, not a population confidence interval or statistical significance test. The same seed labels do not guarantee equal actual initial parameter values in different architectures.

In [ ]:
# ==================== 07. Paired Seed/Fold Stability Analysis ====================

# 1. Isolate the two architectures under their same fixed weighted CE loss.
stability=fold_results.loc[fold_results["loss_method"]=="weighted_ce"]
display(stability.pivot_table(
    index=["seed","fold"],columns="architecture",values="macro_f1"
).round(4))
seed_summary=stability.groupby(["architecture","seed"],as_index=False).agg(
    mean_macro_f1=("macro_f1","mean"),
    fold_sd=("macro_f1","std"),
)
display(seed_summary.round(4))

# 2. Visualise seed-level generalisation variation with shared axis scaling.
fig,ax=plt.subplots(figsize=(10,5))
for arch,frame in seed_summary.groupby("architecture"):
    ax.plot(frame["seed"].astype(str),frame["mean_macro_f1"],marker="o",label=arch)
ax.set(xlabel="Reproducibility seed",ylabel="Mean three-fold Macro F1",
       title="Model architecture stability under matched development folds",ylim=(0,1))
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
plt.show()

**08. Loss Ablation: Weighted CE, Focal and Class-Balanced CE**

This experiment fixes the 1D-ResNet architecture, source windows, three Fold partitions, seed values, optimiser-update budget and basic training configuration. Changing only the class-aware loss does not perfectly isolate a physiological cause; it tests which loss formulation behaves better within this fixed modelling pipeline. Compare each method for the *same seed and fold*, not by selecting the maximum across nine development results.

In [ ]:
# ==================== 08. Fully Paired ResNet Class-Imbalance Ablation ====================

# 1. Compare loss functions over identical seed-by-fold blocks.
resnet=fold_results[fold_results["architecture"]=="1D-ResNet"]
paired_loss=resnet.pivot(index=["seed","fold"],columns="loss_method",values="macro_f1")
display(paired_loss.round(4))
paired_loss["focal_minus_weighted"]=paired_loss["focal"]-paired_loss["weighted_ce"]
paired_loss["class_balanced_minus_weighted"]=(
    paired_loss["class_balanced"]-paired_loss["weighted_ce"]
)
display(paired_loss[[
    "focal_minus_weighted","class_balanced_minus_weighted"
]].round(4))

# 2. Display distribution of paired differences without p-value overclaiming.
fig,ax=plt.subplots(figsize=(11,5))
paired_loss[[
    "focal_minus_weighted","class_balanced_minus_weighted"
]].plot(kind="bar",ax=ax)
ax.axhline(0,color="black",linewidth=.8)
ax.set(title="Paired ResNet loss changes vs weighted CE",
       xlabel="Seed and held-out fold",ylabel="Macro F1 difference")
ax.grid(axis="y",alpha=0.2)
fig.tight_layout()
plt.show()

**09. Shuffling and Stair Activities Across Seeds and Loss Choices**

For low-support classes, inspect actual pooled out-of-fold precision, recall, F1 and class support for **each model/loss/seed condition**. Never use the best seed in isolation as the headline result. Changes in Shuffling recall may be accompanied by lower precision or worse performance on other classes. Stairs ascending and descending must remain separate and retain documented original labels.

In [ ]:
# ==================== 09. Minority-Class Outcomes with Genuine Support ====================

# 1. Keep the three main underrepresented source classes separate.
minority=class_results.loc[class_results["class_code"].isin([3,4,5])].copy()
minority["activity"]=minority["class_code"].map(ACTIVITY_NAMES)
display(minority[[
    "architecture","loss_method","seed","activity",
    "support","precision","recall","f1"
]].round(4))

# 2. Compare Shuffling across all ResNet training-loss conditions.
shuffle=minority.loc[
    (minority["architecture"]=="1D-ResNet") & (minority["class_code"]==3)
]
fig,ax=plt.subplots(figsize=(11,5))
for method,frame in shuffle.groupby("loss_method"):
    ax.plot(frame["seed"].astype(str),frame["f1"],marker="o",label=method)
ax.set(xlabel="Random seed",ylabel="Pooled out-of-fold Shuffling F1",
       title="ResNet minority-class sensitivity to training loss",ylim=(0,1))
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
plt.show()

**10. Equal-Person Results and Computational Cost**

Because different people contribute different numbers of overlapping windows, report the average of **one fixed-eight-class Macro F1 per held-out person** alongside the window-level fold metrics. Training time depends on GPU load and model complexity and should not be used as a surrogate for algorithmic FLOPs. Cached results distinguish actual complete model-fit times from a zero-time restore; no model-selection conclusion is based on these cost measurements alone.

In [ ]:
# ==================== 10. Person-Wise Robustness and Resource Consumption ====================

# 1. Summarise genuine person-level development results per condition.
people_summary=subject_results.groupby(
    ["architecture","loss_method","seed"],as_index=False
).agg(
    mean_person_macro_f1=("macro_f1_fixed_task","mean"),
    person_macro_f1_sd=("macro_f1_fixed_task","std")
)
display(people_summary.round(4))

# 2. Display training-time and fold-metric spread across all trials.
compute=fold_results.groupby(
    ["architecture","loss_method","seed"],as_index=False
).agg(
    mean_fold_macro_f1=("macro_f1","mean"),
    fold_macro_f1_sd=("macro_f1","std"),
    total_train_s=("train_seconds","sum"),
    total_inference_s=("inference_seconds","sum"),
)
display(compute.round(3))

**11. Research Interpretation and Model-Selection Restrictions**

The next stage (25) will freeze a model and all preprocessing decisions using Development results **only**. This Stage 24 robustness study informs that decision but is not an unbiased, independent estimate of a selected model's final performance. Any method that helps mean Macro F1 but deteriorates Shuffling, stairs or minority precision should be described accurately. Focal Loss is not inherently better; class-balanced weighting is based on correlated overlapping windows, so its nominal count assumptions are an approximation.

In [ ]:
# ==================== 11. Evidence-Bounded Robustness Summary ====================

# 1. Report all predeclared conditions rather than cherry-pick favourable seeds.
final_summary=compute.groupby(
    ["architecture","loss_method"],as_index=False
).agg(
    mean_across_seed_fold_macro_f1=("mean_fold_macro_f1","mean"),
    across_seed_sd=("mean_fold_macro_f1","std"),
    total_training_s=("total_train_s","sum"),
)
display(final_summary.sort_values(
    "mean_across_seed_fold_macro_f1",ascending=False
).round(4))

# 2. Document the primary remaining scientific limitations.
limitations=pd.DataFrame([
    {"issue":"Development-only estimates","boundary":"No independent held-out Test claim"},
    {"issue":"Three repeated seeds","boundary":"Limited stochastic robustness evidence"},
    {"issue":"Overlapping source windows","boundary":"Windows are not independent observations"},
    {"issue":"Approximate class counts","boundary":"Class-balanced effective number assumes counts as proxies"},
    {"issue":"Loss selection","boundary":"Select on Development only; disclose post-hoc comparisons"},
    {"issue":"Original Test historical inspection","boundary":"S023/S032 not strictly preregistered-unseen"},
])
display(limitations)

**12. Final Quality Gate and Stage 25 Handover**

The Quality Gate verifies the complete 36-job experiment grid; identical source windows and Fold assignments; proper Train-only class weights; complete per-person coverage; real class-level support; and no protected Test participants. It does not establish a final model or independent generalisation. After reviewing actual outputs, Stage 25 will predeclare and freeze the final modelling protocol before any reserved-Test scoring or creation of deployable artefacts.

In [ ]:
# ==================== 12. Scientific Robustness Quality Gate ====================

# 1. Verify every model/loss/seed/fold job was completed exactly once.
assert len(jobs)==36
assert len(fold_results)==36
assert not fold_results[["architecture","loss_method","seed","fold"]].duplicated().any()
assert len(subject_results)==29*len(EXPERIMENTS)
assert len(class_results)==8*len(EXPERIMENTS)

# 2. Verify all conditions use the complete original development source rows.
assert len(predictions)==14201*len(EXPERIMENTS)
for key,table in predictions.groupby(["architecture","loss_method","seed"]):
    assert len(table)==len(y) and table["source_row"].is_unique
    assert not(set(table["subject_id"])&RESERVED_TEST)
    np.testing.assert_array_equal(
        table.sort_values("source_row")["true_label"].to_numpy(),y
    )
assert fold_results["macro_f1"].between(0,1).all()

# 3. Show clear completion and scientific boundaries in a numbered summary.
print("\n"+"="*70)
print("STAGE 24 | MULTI-SEED AND CLASS-IMBALANCE ROBUSTNESS")
print("="*70)
print("Distinct model/loss/seed conditions:",len(EXPERIMENTS))
print("Complete model/fold jobs:",len(jobs))
print("Development participants represented:",subject_results["subject_id"].nunique())
print("Per-condition held-out source windows:",len(y))
print("Loss and seed controls: VERIFIED")
print("Original protected Test participants used: NO")
print("-"*70)
print("Stage 24 scientific quality gate: PASSED")
print("Final model selection performed: NO")
print("Independent generalisation established: NO")
print("Next stage: Stage 25 - Final Model Selection and Protocol Freeze")
print("="*70)